# Setting up the DuckDB Python Client

This notebook contains the code examples from chapter 7 of *Getting Started with DuckDB*.

## Technical requirements

### Cloning the repository

Install Git from https://git-scm.com/downloads

Instructions on how to clone repositories from GitHub:
https://docs.github.com/en/repositories/creating-and-managing-repositories/cloning-a-repository 


#### Installing Python

Several ways to get a local Python installation on your machine:
- https://www.python.org/downloads
- https://docs.python.org/3/using
- https://www.anaconda.com/download

### Creating a Python virtual environment

For a more comprehensive guide, see the Python documentation for the **venv** module: https://docs.python.org/3/library/venv.html

In [ ]:
! python -m venv duckdb-book

#### Activating the Virtual Environment:

#### Linux Environment

In [ ]:
! source duckdb-book/bin/activate

#### Windows Environment

#### cmd.exe:

In [ ]:
! duckdb-book\Scripts\activate.bat

#### Windows Powershell:

In [ ]:
! Set-ExecutionPolicy -ExecutionPolicy RemoteSigned -Scope CurrentUser # Windows Powershell
! .\duckdb-book\Scripts\Activate.ps1

#### To deactivate the virtual environment, run the following command:

In [ ]:
! deactivate

In order to run the examples in this notebook, you'll need to install the Python dependencies for this project. You can do this by running the following command in your terminal when in the root directory of the project. Note that ideally this should be using a Python virtual environment for this project.

### Installing the Python dependencies

In [ ]:
! pip install -r requirements.txt

In [ ]:
! pip install duckdb emoji ibis-framework jupysql jupyter-lsp jupyterlab pandas plotly polars pyarrow sqlparse

For complete instructions on how to set up your environment for working through the examples, please consult the *Technical requirements* section of this chapter in the book.

### Choosing your Python IDE

Use one of the following IDEs for working with Jupyter Notebooks:
- JupyterLab
- Microsoft **Visual Studio Code (VS Code)**

To start JupyterLab, simply run this command in your terminal:

In [ ]:
! jupyter-lab

If you prefer the classic Jupyter Notebook experience rather than JupyterLab, you can launch Jupyter Notebook by running the below command in your terminal:

In [ ]:
! jupyter-notebook

Alternatively, if you already use and familiar with VS Code, you may find that you prefer to use its Jupyter Notebook support. You can find instructions on how to set this up in the VS Code documentation: https://code.visualstudio.com/docs/datascience/jupyter-notebooks. Note that VS Code's Jupyter Notebook support sitll requires a Jupyter installation, which can be sourced from the **duckdb-book** virtual environment. 

## Connecting to DuckDB in Python

### The default in-memory database 

**Using the default in-memory database:** This database is automatically created for you when you import the duckdb module. The default database provides a convenient way to quickly perform activities such as ad-hoc data analysis when you know you will only need a single database in your session. This is what you will connect to when you invoke a function from the duckdb module that interacts with a database, such as **duckdb.sql()** and **duckdb.execute()**.

In [1]:
import duckdb

duckdb.sql("SELECT 'duck' AS animal, 'quack!' AS greeting")

┌─────────┬──────────┐
│ animal  │ greeting │
│ varchar │ varchar  │
├─────────┼──────────┤
│ duck    │ quack!   │
└─────────┴──────────┘

If you are wondering about the two different types of quote characters in the above code, remember that in DuckDB SQL, literal string values are represented using single-quote characters. The outer double-quote characters define the Python string literal containing our SQL query that we want to send to the database. As Python can use either double quotes or single quotes to represent string literals, we recommend using double quotes for Python string literals that contain SQL queries.

The result of the **sql()** method call is a DuckDB **relation** object, which represents the query.

The default in-memory database is particularly convenient for interactive data analysis workflows, as you can jump in and start working with DuckDB without having to set up any connections. If you're doing ad-hoc data wrangling or exploratory data analysis in a notebook, you may often find yourself reaching for this approach. In other scenarios, the default database won't always be the most appropriate way to work with DuckDB. Being an in-memory database, the default database is ephemeral, meaning that its contents will be lost when its parent Python process ends. If you want to persist your database state after the process ends, you'll need to create a new persistent connection explicitly, rather than use the default database.

### Managing database connections explicitly 

**Explicitly creating and managing database connections:** This gives you more flexibility and control over how you work with DuckDB databases. Database connection objects are created via the **duckdb.connect()** function, which you then use to issue commands against the database it's connected to.

In [2]:
conn = duckdb.connect()

conn.sql(
    """
    CREATE TABLE hello AS
    SELECT 'pato' AS animal, 'cuac!' AS greeting
    """
)

conn.sql("SELECT * FROM hello")

┌─────────┬──────────┐
│ animal  │ greeting │
│ varchar │ varchar  │
├─────────┼──────────┤
│ pato    │ cuac!    │
└─────────┴──────────┘

The **connect()** function returns a **DuckDBPyConnection** object that represents a connection to a specific DuckDB database. Calling **connect()** without any arguments results in the creation of a new in-memory database.

The **connect()** function has several parameters, the first of which is the **database** parameter, which takes a string specifying the database you want to connect to.

In [3]:
conn = duckdb.connect(database=":memory:")

If you happen to want to create an explicit connection object to the default in-memory database, you can pass the **:default:** string as the value of the **database** keyword argument:

In [ ]:
default_connection = duckdb.connect(database=":default:")

Explicitly creating connection objects also allows us to specify database configuration options. To do this, you need to pass a dictionary of configuration key-value pairs to the **connect()** function's **config** parameter. Here's an example where we create a new in-memory database, which we configure to use a maximum of 10 GB of system memory, as well as limiting the database to using four CPU threads for parallel query execution:

In [4]:
custom_conn = duckdb.connect(
    config={
        "memory_limit": "10GB", 
        "threads": 1
    }
)

See the DuckDB performance guide more information around tuning DuckDB's performance:
https://duckdb.org/docs/guides/performance

For the complete list of database configuration options that can be provided via the **config** parameter of the **connect** function, consult the DuckDB configuration documentation: https://duckdb.org/docs/configuration.

### Connections to persistent-storage databases

In [5]:
conn = duckdb.connect(database="quack.duckdb")

conn.sql(
    """
    CREATE OR REPLACE TABLE hello AS
    SELECT 'ente' AS animal, 'quak!' AS greeting
    """
)

conn.close()

In [6]:
! duckdb quack.duckdb -c 'SELECT * FROM hello'

┌─────────┬──────────┐
│ animal  │ greeting │
│ varchar │ varchar  │
├─────────┼──────────┤
│ ente    │ quak!    │
└─────────┴──────────┘


### Closing database connections 

A convenient way to ensure that you close your connections is to use connection objects as **context managers**. Context managers are Python objects that can be used with the **with** keyword to automatically perform certain actions while entering and exiting the **with** statement's code block. They are commonly used for managing external resources such as database connections, network connections, and file descriptors. Here's an example of using a DuckDB connection object as a context manager:

In [7]:
sql = "INSERT INTO hello VALUES ('Labradorius', 'quack!')" 

with duckdb.connect(database="quack.duckdb") as conn:
    conn.sql(sql) 

The **as** keyword allows us to capture the connection object as a variable, for reference inside the indented code block of the **with** statement. When the code block is exited, the connection's **close()** method will be called automatically for us. This provides a convenient way to create a new connection and also automate its closing, freeing you from having to remember to explicitly close every connection you create.

### Sharing disk-based databases between processes

Persistent disk-based DuckDB databases can be shared across multiple processes, provided that all databases are connected in read-only mode. You can specify that a database is opened in read-only mode by giving the **connect()** function's **read_only** parameter the value **True**:

In [8]:
conn = duckdb.connect(database="quack.duckdb", read_only=True)
# query the database in here...
conn.close() 

### Installing and loading extensions 

In [9]:
duckdb.install_extension("spatial")  

duckdb.load_extension("spatial")

In [10]:
duckdb.sql(
    """
    SELECT *
    FROM duckdb_extensions()
    WHERE loaded = true
    """
)

┌────────────────┬─────────┬───────────┬────────────────────────────────────────────────────────────────────────────────┬────────────────────────────────────────────────────────────────────────────────────┬───────────┬───────────────────┬───────────────────┬────────────────┐
│ extension_name │ loaded  │ installed │                                  install_path                                  │                                    description                                     │  aliases  │ extension_version │   install_mode    │ installed_from │
│    varchar     │ boolean │  boolean  │                                    varchar                                     │                                      varchar                                       │ varchar[] │      varchar      │      varchar      │    varchar     │
├────────────────┼─────────┼───────────┼────────────────────────────────────────────────────────────────────────────────┼───────────────────────────────────────────────────

In [11]:
conn = duckdb.connect()
conn.sql(
    """
    SELECT *
    FROM duckdb_extensions()
    WHERE loaded = true
    """
)

┌────────────────┬─────────┬───────────┬──────────────┬──────────────────────────────────────────────────────────────────┬───────────┬───────────────────┬───────────────────┬────────────────┐
│ extension_name │ loaded  │ installed │ install_path │                           description                            │  aliases  │ extension_version │   install_mode    │ installed_from │
│    varchar     │ boolean │  boolean  │   varchar    │                             varchar                              │ varchar[] │      varchar      │      varchar      │    varchar     │
├────────────────┼─────────┼───────────┼──────────────┼──────────────────────────────────────────────────────────────────┼───────────┼───────────────────┼───────────────────┼────────────────┤
│ core_functions │ true    │ true      │ (BUILT-IN)   │ Core function library                                            │ []        │ v1.4.0            │ STATICALLY_LINKED │                │
│ icu            │ true    │ true      │

In [12]:
conn.install_extension("spatial")  

conn.load_extension("spatial")

This also serves as a reminder that multiple DuckDB databases running in a Python process are distinct and do not share state with each other.

## Summary